<a href="https://colab.research.google.com/github/mafloan/Agente-IA-BimBam-Buy/blob/main/Agente_IA_BimBamBuy_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import sys
!{sys.executable} -m pip install -q langchain-community pypdf fpdf2 cohere faiss-cpu langchain-google-genai langchain-text-splitters ipywidgets
print('✅ Dependencies installed')

✅ Dependencies installed


In [ ]:
import os
import shutil
from pathlib import Path

# Mount Google Drive (already mounted, but confirming)
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

# Your PDF folder ID from the Google Drive link
FOLDER_PATH = '/content/drive/MyDrive'

# PDF file names to search for
PDF_NAMES = [
    'Guia de tiempos y costos.pdf',
    'Manual de Garantia.pdf',
    'Politica de reembolsos.pdf',
    'Preguntas frecuentes.pdf',
    'Programa de afiliados.pdf',
]

print("Looking for PDFs in Google Drive...\n")

# Search for PDFs in the mounted drive
found_files = []
for root, dirs, files in os.walk(FOLDER_PATH):
    for file in files:
        if file.endswith('.pdf'):
            full_path = os.path.join(root, file)
            print(f"Found: {file}")
            print(f"  Path: {full_path}")

            # Copy to /content/data
            dest_path = os.path.join(DATA_DIR, file)
            shutil.copy(full_path, dest_path)
            print(f"  ✅ Copied to: {dest_path}\n")
            found_files.append(file)

# Create FILES list
FILES = [
    (os.path.join(DATA_DIR, filename), filename.replace('.pdf', '').lower())
    for filename in found_files
]

print(f'✅ Found and copied {len(found_files)} PDFs')
print(f'Total files ready: {len(FILES)}')








Mounted at /content/drive
Looking for PDFs in Google Drive...

Found: Certificado_2174075060103706.pdf
  Path: /content/drive/MyDrive/Certificado_2174075060103706.pdf
  ✅ Copied to: /content/data/Certificado_2174075060103706.pdf

Found: Motivation Letter Farias Franzosi.pdf
  Path: /content/drive/MyDrive/Motivation Letter Farias Franzosi.pdf
  ✅ Copied to: /content/data/Motivation Letter Farias Franzosi.pdf

Found: María Florencia Farías Franzosi resume 2020.pdf
  Path: /content/drive/MyDrive/María Florencia Farías Franzosi resume 2020.pdf
  ✅ Copied to: /content/data/María Florencia Farías Franzosi resume 2020.pdf

Found: MFFF TITULO.pdf
  Path: /content/drive/MyDrive/MFFF TITULO.pdf
  ✅ Copied to: /content/data/MFFF TITULO.pdf

Found: Rosana's lesson (njx-aoym-fmo – 4 May 2021).pdf
  Path: /content/drive/MyDrive/Rosana's lesson (njx-aoym-fmo – 4 May 2021).pdf
  ✅ Copied to: /content/data/Rosana's lesson (njx-aoym-fmo – 4 May 2021).pdf

Found: cv María Florencia Farías Franzos

In [ ]:
import os
from fpdf import FPDF

DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)

FILES = [
    (os.path.join(DATA_DIR, 'Guia de tiempos y costos.pdf'), 'envios'),
    (os.path.join(DATA_DIR, 'Manual de Garantia.pdf'), 'garantia'),
    (os.path.join(DATA_DIR, 'Politica de reembolsos.pdf'), 'reembolsos'),
    (os.path.join(DATA_DIR, 'Preguntas frecuentes.pdf'), 'faq'),
    (os.path.join(DATA_DIR, 'Programa de afiliados.pdf'), 'afiliados'),
]

def create_dummy_pdf(file_path, content):
    if not os.path.exists(file_path):
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font('Arial', size=12)
        pdf.multi_cell(0, 10, txt=content)
        pdf.output(file_path)
        print(f'Created: {file_path}')

for path, doc_type in FILES:
    content = f'Documentation for {doc_type}. Important policies and information about {doc_type}.'
    create_dummy_pdf(path, content)

print(f'✅ Files ready in: {DATA_DIR}')

✅ Files ready in: /content/data


In [ ]:
from langchain_community.document_loaders import PyPDFLoader

def load_documents():
    all_docs = []
    for path, doc_type in FILES:
        try:
            loader = PyPDFLoader(path)
            docs = loader.load()  # This loads ALL pages, not just the first

            # Add metadata to each document
            for d in docs:
                d.metadata['source_doc'] = doc_type
                d.metadata['file_name'] = os.path.basename(path)

            all_docs.extend(docs)
            print(f'✅ Loaded {len(docs)} pages from {doc_type}')
        except Exception as e:
            print(f'❌ Error loading {path}: {e}')

    print(f'\n📄 Total pages loaded: {len(all_docs)}')
    return all_docs

print('✅ load_documents() defined - loads ALL pages from PDFs')



✅ load_documents() defined - loads ALL pages from PDFs


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_documents(documents):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=700,
        chunk_overlap=120,
        separators=['\n\n', '\n', '.', ' ']
    )
    chunks = splitter.split_documents(documents)
    print(f'Split into {len(chunks)} chunks')
    return chunks

print('✅ split_documents() defined')

✅ split_documents() defined


In [ ]:
from langchain_community.embeddings import CohereEmbeddings

def create_embeddings(api_key):
    try:
        embeddings = CohereEmbeddings(
            cohere_api_key=api_key,
            model='embed-multilingual-v3.0',
            user_agent='langchain'
        )
        print('Embeddings initialized')
        return embeddings
    except Exception as e:
        print(f'Error: {e}')
        return None

print('✅ create_embeddings() defined')

✅ create_embeddings() defined


In [ ]:
from langchain_community.vectorstores import FAISS

def create_vectorstore(chunks, embeddings):
    texts = [doc.page_content for doc in chunks]
    metadatas = [doc.metadata for doc in chunks]
    vectorstore = FAISS.from_texts(texts, embeddings, metadatas=metadatas)
    print(f'Vectorstore created with {len(chunks)} vectors')
    return vectorstore

print('✅ create_vectorstore() defined')

✅ create_vectorstore() defined


In [ ]:
def create_retriever(vectorstore):
    retriever = vectorstore.as_retriever(
        search_type='similarity',
        search_kwargs={'k': 5}
    )
    print('Retriever created (k=5)')
    return retriever

print('✅ create_retriever() defined')

✅ create_retriever() defined


In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

def create_rag_chain(retriever, google_api_key):
    llm = ChatGoogleGenerativeAI(
        model='gemini-3.5-flash',
        google_api_key=google_api_key,
        temperature=0.2
    )

    prompt = ChatPromptTemplate.from_messages([
        ('system', 'You are an expert on BimBam Buy policies. Use ONLY the provided context. Rules: Answer in Spanish. If insufficient info, say: No tengo informacion suficiente. Be clear and direct. Context: {context}'),
        ('human', '{question}')
    ])

    def process_question(input_dict):
        question = input_dict['question']
        docs = retriever.invoke(question)
        context = '\n\n'.join([d.page_content for d in docs])

        chain = prompt | llm | StrOutputParser()
        answer = chain.invoke({'context': context, 'question': question})

        sources = [{'doc': d.metadata['source_doc'], 'file': d.metadata['file_name']} for d in docs]

        return {'answer': answer, 'sources': sources}

    return process_question

print('✅ create_rag_chain() defined')



✅ create_rag_chain() defined


In [ ]:
from google.colab import userdata

COHERE_API_KEY = userdata.get('COHERE_API_KEY')
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')

if not COHERE_API_KEY:
    raise ValueError('COHERE_API_KEY not found in Secrets')
if not GOOGLE_API_KEY:
    raise ValueError('GOOGLE_API_KEY not found in Secrets')

print('✅ API keys loaded')

✅ API keys loaded


In [ ]:
print('Building RAG pipeline...\n')

docs = load_documents()
print()

chunks = split_documents(docs)
print()

embeddings = create_embeddings(COHERE_API_KEY)
print()

vectorstore = create_vectorstore(chunks, embeddings)
print()

retriever = create_retriever(vectorstore)
print()

rag_chain = create_rag_chain(retriever, GOOGLE_API_KEY)

print('\n' + '='*50)
print('✅ RAG Pipeline Ready!')
print('='*50)





Building RAG pipeline...

✅ Loaded 1 pages from certificado_2174075060103706
✅ Loaded 2 pages from motivation letter farias franzosi
✅ Loaded 2 pages from maría florencia farías franzosi resume 2020
✅ Loaded 2 pages from mfff titulo
✅ Loaded 1 pages from rosana's lesson (njx-aoym-fmo – 4 may 2021)
✅ Loaded 3 pages from cv maría florencia farías franzosi mayo 2021
✅ Loaded 1 pages from fff (10)
✅ Loaded 1 pages from farias franzosi octubre 2021
✅ Loaded 1 pages from cv_farias franzosi_florencia
✅ Loaded 1 pages from cv_farias franzosi_florencia (4)
❌ Error loading /content/data/CV_Farias Franzosi_Florencia (3).pdf: Cannot read an empty file
✅ Loaded 1 pages from cv_farias franzosi_florencia (2)
✅ Loaded 1 pages from cv_farias franzosi_florencia (1)
✅ Loaded 1 pages from cv_farias franzosi_florencia
✅ Loaded 1 pages from farías franzosi rse (2)
✅ Loaded 1 pages from farías franzosi rse (1)
✅ Loaded 1 pages from farías franzosi rse
✅ Loaded 1 pages from cv_fariasfranzosi_mariaflore

✅ Loaded 2 pages from dni mfff
✅ Loaded 1 pages from cross media content seminar
✅ Loaded 1 pages from digital marketing and media diploma
✅ Loaded 4 pages from employment story 
✅ Loaded 1 pages from international fundraising congress
✅ Loaded 1 pages from playersvocabularytask
✅ Loaded 1 pages from playersvocabulary
✅ Loaded 1 pages from fariasfranzosi interpreter 2025
✅ Loaded 1 pages from certificado_2175976777604982
✅ Loaded 1 pages from fa005000262495
✅ Loaded 1 pages from certificado_2175985390697440
✅ Loaded 12 pages from guia de tiempos y costos
✅ Loaded 14 pages from politica de reembolsos
✅ Loaded 11 pages from programa de afiliados
✅ Loaded 10 pages from preguntas frecuentes
✅ Loaded 10 pages from manual de garantia
✅ Loaded 1 pages from farias franzosi resume 22
✅ Loaded 10 pages from fym_5_face_yoga_poses
✅ Loaded 6 pages from fl378850
✅ Loaded 3 pages from 27311603774_011_00001_00000036
✅ Loaded 1 pages from factura pizarraid432440

📄 Total pages loaded: 124

Split into 

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

question_input = widgets.Text(
    placeholder='Ask about BimBam Buy policies...',
    description='Q:',
    layout=widgets.Layout(width='70%')
)

send_button = widgets.Button(
    description='Ask',
    button_style='info',
    icon='search'
)

output_area = widgets.Output()

def on_send_click(b):
    question = question_input.value.strip()

    if not question:
        with output_area:
            clear_output()
            print('⚠️ Please enter a question')
        return

    question_input.disabled = True
    send_button.disabled = True

    with output_area:
        clear_output()
        print('Processing...')

        try:
            result = rag_chain({'question': question})
            clear_output()
            print('RESPONSE:')
            print('-' * 60)
            print(result['answer'])
            print('-' * 60)
            print('\nSources:')
            sources_set = set()
            for src in result['sources']:
                sources_set.add(src['file'])
            for s in sorted(sources_set):
                print(f'  • {s}')
        except Exception as e:
            clear_output()
            print(f'Error: {str(e)}')
            import traceback
            traceback.print_exc()

    question_input.disabled = False
    send_button.disabled = False
    question_input.value = ''

send_button.on_click(on_send_click)

input_box = widgets.HBox([question_input, send_button])
interface = widgets.VBox([input_box, output_area])

print('Chat ready! Ask questions below:')
display(interface)







Chat ready! Ask questions below:


In [ ]:
test_question = 'What are the main policies?'
print(f'Test Question: {test_question}\n')

try:
    result = rag_chain.invoke({'question': test_question})
    print('Answer:')
    print(result['answer'])
    print('\nSources used:')
    for src in result['sources']:
        print(f'  • {src["file"]}')
except Exception as e:
    print(f'❌ Error during test: {e}')
    print('The RAG chain may not be fully initialized yet.')
    print('Try using the interactive chat interface above instead.')


Test Question: What are the main policies?

❌ Error during test: 'function' object has no attribute 'invoke'
The RAG chain may not be fully initialized yet.
Try using the interactive chat interface above instead.


In [ ]:
import google.generativeai as genai

GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

# List available models
print("Available models:")
for model in genai.list_models():
    print(f"  - {model.name}")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Available models:
  - models/gemini-2.5-flash
  - models/gemini-2.5-pro
  - models/gemini-2.0-flash
  - models/gemini-2.0-flash-001
  - models/gemini-2.0-flash-lite-001
  - models/gemini-2.0-flash-lite
  - models/gemini-2.5-flash-preview-tts
  - models/gemini-2.5-pro-preview-tts
  - models/gemma-4-26b-a4b-it
  - models/gemma-4-31b-it
  - models/gemini-flash-latest
  - models/gemini-flash-lite-latest
  - models/gemini-pro-latest
  - models/gemini-2.5-flash-lite
  - models/gemini-2.5-flash-image
  - models/gemini-3-pro-preview
  - models/gemini-3-flash-preview
  - models/gemini-3.1-pro-preview
  - models/gemini-3.1-pro-preview-customtools
  - models/gemini-3.1-flash-lite-preview
  - models/gemini-3.1-flash-lite
  - models/gemini-3-pro-image-preview
  - models/gemini-3-pro-image
  - models/nano-banana-pro-preview
  - models/gemini-3.1-flash-image-preview
  - models/gemini-3.1-flash-image
  - models/gemini-3.1-flash-lite-image
  - models/gemini-3.5-flash
  - models/gemini-3.5-flash-lite
  

In [ ]:
print("=== DIAGNOSTIC CHECK ===\n")

# Check what files are in /content/data
print("Files in /content/data:")
for file in os.listdir(DATA_DIR):
    file_path = os.path.join(DATA_DIR, file)
    size = os.path.getsize(file_path)
    print(f"  • {file} ({size} bytes)")

print("\n" + "="*50)
print("FILES list:")
for path, doc_type in FILES:
    print(f"  • {doc_type}: {path}")

print("\n" + "="*50)
print("Loading documents...")
docs = load_documents()

print("\n" + "="*50)
if len(docs) > 0:
    print("Sample of first document:")
    print(f"  Length: {len(docs[0].page_content)} characters")
    print(f"  Content preview:\n{docs[0].page_content[:200]}...\n")
else:
    print("❌ NO DOCUMENTS LOADED!")

=== DIAGNOSTIC CHECK ===

Files in /content/data:
  • 27311603774_011_00001_00000036.pdf (55147 bytes)
  • Farias Franzosi octubre 2021.pdf (605436 bytes)
  • FL378850.pdf (167460 bytes)
  • CV_FariasFranzosi_MariaFlorencia (mkt).pdf (171061 bytes)
  • CV_Farias Franzosi_Florencia (1).pdf (607689 bytes)
  • CV_Farias Franzosi_Florencia (2).pdf (607689 bytes)
  • FA005000262495.pdf (58754 bytes)
  • CV_FariasFranzosi2022.pdf (190367 bytes)
  • cv María Florencia Farías Franzosi mayo 2021.pdf (234783 bytes)
  • Guia de tiempos y costos.pdf (294155 bytes)
  • CV_FariasFranzosi2022 (4).pdf (190367 bytes)
  • Farías Franzosi RSE (1).pdf (608550 bytes)
  • CV_Farias Franzosi_Florencia (4).pdf (607689 bytes)
  • CV_FariasFranzosi_MariaFlorencia (mkt) (1).pdf (171061 bytes)
  • Certificado_2175985390697440.pdf (110132 bytes)
  • Cv_Farias Franzosi_Florencia.pdf (609420 bytes)
  • CV_Farias Franzosi_Florencia.pdf (607689 bytes)
  • FariasFranzosi Interpreter 2025.pdf (89897 bytes)
  • CV_Far

✅ Loaded 2 pages from dni mfff
✅ Loaded 1 pages from cross media content seminar
✅ Loaded 1 pages from digital marketing and media diploma
✅ Loaded 4 pages from employment story 
✅ Loaded 1 pages from international fundraising congress
✅ Loaded 1 pages from playersvocabularytask
✅ Loaded 1 pages from playersvocabulary
✅ Loaded 1 pages from fariasfranzosi interpreter 2025
✅ Loaded 1 pages from certificado_2175976777604982
✅ Loaded 1 pages from fa005000262495
✅ Loaded 1 pages from certificado_2175985390697440
✅ Loaded 12 pages from guia de tiempos y costos
✅ Loaded 14 pages from politica de reembolsos
✅ Loaded 11 pages from programa de afiliados
✅ Loaded 10 pages from preguntas frecuentes
✅ Loaded 10 pages from manual de garantía
✅ Loaded 1 pages from farias franzosi resume 22
✅ Loaded 10 pages from fym_5_face_yoga_poses
✅ Loaded 6 pages from fl378850
✅ Loaded 3 pages from 27311603774_011_00001_00000036
✅ Loaded 1 pages from factura pizarraid432440

📄 Total pages loaded: 124

Sample of 

In [ ]:
import os

DATA_DIR = '/content/data'

# ONLY the 5 BimBam Buy PDFs
FILES = [
    (os.path.join(DATA_DIR, 'Guia de tiempos y costos.pdf'), 'envios'),
    (os.path.join(DATA_DIR, 'Manual de Garantía.pdf'), 'garantia'),
    (os.path.join(DATA_DIR, 'Politica de reembolsos.pdf'), 'reembolsos'),
    (os.path.join(DATA_DIR, 'Preguntas frecuentes.pdf'), 'faq'),
    (os.path.join(DATA_DIR, 'Programa de afiliados.pdf'), 'afiliados'),
]

print('✅ FILES list configured with only BimBam Buy PDFs:')
for path, doc_type in FILES:
    size = os.path.getsize(path)
    print(f'  • {doc_type}: {os.path.basename(path)} ({size} bytes)')

✅ FILES list configured with only BimBam Buy PDFs:
  • envios: Guia de tiempos y costos.pdf (294155 bytes)


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/Manual de Garantía.pdf'

In [ ]:
import os

DATA_DIR = '/content/data'

# List all PDF files to see what we have
print("Available PDFs in /content/data:")
all_pdfs = [f for f in os.listdir(DATA_DIR) if f.endswith('.pdf')]
for pdf in sorted(all_pdfs):
    print(f"  • {pdf}")

print("\n" + "="*60)

# The 5 BimBam Buy PDFs - using the exact filenames
FILES = [
    (os.path.join(DATA_DIR, 'Guia de tiempos y costos.pdf'), 'envios'),
    (os.path.join(DATA_DIR, 'Manual de Garantia.pdf'), 'garantia'),  # No accent on "a"
    (os.path.join(DATA_DIR, 'Politica de reembolsos.pdf'), 'reembolsos'),
    (os.path.join(DATA_DIR, 'Preguntas frecuentes.pdf'), 'faq'),
    (os.path.join(DATA_DIR, 'Programa de afiliados.pdf'), 'afiliados'),
]

print('✅ FILES list configured with BimBam Buy PDFs:')
for path, doc_type in FILES:
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f'  ✅ {doc_type}: {os.path.basename(path)} ({size} bytes)')
    else:
        print(f'  ❌ {doc_type}: {os.path.basename(path)} - NOT FOUND')

Available PDFs in /content/data:
  • 27311603774_011_00001_00000036.pdf
  • CV_Farias Franzosi_Florencia (1).pdf
  • CV_Farias Franzosi_Florencia (2).pdf
  • CV_Farias Franzosi_Florencia (3).pdf
  • CV_Farias Franzosi_Florencia (4).pdf
  • CV_Farias Franzosi_Florencia.pdf
  • CV_FariasFranzosi2022 (1).pdf
  • CV_FariasFranzosi2022 (2).pdf
  • CV_FariasFranzosi2022 (3).pdf
  • CV_FariasFranzosi2022 (4).pdf
  • CV_FariasFranzosi2022.pdf
  • CV_FariasFranzosi_MariaFlorencia (mkt) (1).pdf
  • CV_FariasFranzosi_MariaFlorencia (mkt) (2).pdf
  • CV_FariasFranzosi_MariaFlorencia (mkt).pdf
  • CV_FariasFranzosi_MariaFlorencia.pdf
  • Certificado_2174075060103706.pdf
  • Certificado_2175976777604982.pdf
  • Certificado_2175985390697440.pdf
  • Cross media content Seminar.pdf
  • Cv_Farias Franzosi_Florencia.pdf
  • DNI MFFF.pdf
  • Digital Marketing and Media Diploma.pdf
  • FA005000262495.pdf
  • FARIAS FRANZOSI RESUME 22.pdf
  • FARIAS+FRANZOSI+RESUME+22.pdf
  • FFF (10).pdf
  • FL378850.pdf
 

In [ ]:
import os

DATA_DIR = '/content/data'

# The 5 BimBam Buy PDFs - using the correct file with the accent
FILES = [
    (os.path.join(DATA_DIR, 'Guia de tiempos y costos.pdf'), 'envios'),
    (os.path.join(DATA_DIR, 'Manual de Garantía.pdf'), 'garantia'),  # With accent - has real content (265KB)
    (os.path.join(DATA_DIR, 'Politica de reembolsos.pdf'), 'reembolsos'),
    (os.path.join(DATA_DIR, 'Preguntas frecuentes.pdf'), 'faq'),
    (os.path.join(DATA_DIR, 'Programa de afiliados.pdf'), 'afiliados'),
]

print('✅ FILES list configured with BimBam Buy PDFs:')
for path, doc_type in FILES:
    size = os.path.getsize(path)
    print(f'  ✅ {doc_type}: {os.path.basename(path)} ({size} bytes)')

print(f'\nTotal: {sum(os.path.getsize(p[0]) for p in FILES)} bytes')

✅ FILES list configured with BimBam Buy PDFs:
  ✅ envios: Guia de tiempos y costos.pdf (294155 bytes)


FileNotFoundError: [Errno 2] No such file or directory: '/content/data/Manual de Garantía.pdf'

In [ ]:
import os
import shutil

DATA_DIR = '/content/data'

# List of ONLY BimBam Buy files to keep
KEEP_FILES = {
    'Guia de tiempos y costos.pdf',
    'Manual de Garantia.pdf',
    'Politica de reembolsos.pdf',
    'Preguntas frecuentes.pdf',
    'Programa de afiliados.pdf',
}

# Remove all other files
print("Cleaning up personal files...\n")
for file in os.listdir(DATA_DIR):
    if file not in KEEP_FILES:
        file_path = os.path.join(DATA_DIR, file)
        try:
            os.remove(file_path)
            print(f"❌ Deleted: {file}")
        except Exception as e:
            print(f"⚠️ Error deleting {file}: {e}")

print("\n" + "="*60)
print("✅ Cleanup complete!\n")
print("Remaining files:")
for file in sorted(os.listdir(DATA_DIR)):
    size = os.path.getsize(os.path.join(DATA_DIR, file))
    print(f"  ✅ {file} ({size} bytes)")

Cleaning up personal files...


✅ Cleanup complete!

Remaining files:
  ✅ Guia de tiempos y costos.pdf (294155 bytes)
  ✅ Politica de reembolsos.pdf (322515 bytes)
  ✅ Preguntas frecuentes.pdf (290388 bytes)
  ✅ Programa de afiliados.pdf (96131 bytes)
